# Libraries

In [11]:
from typing import Optional, Literal, Generator
from copy import deepcopy

import gymnasium as gym
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical, Normal, RelaxedOneHotCategorical
from torch.optim import Optimizer, Adam, AdamW

## Constants 

In [12]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cpu'

## ReplayBuffer: memory allocation and management

### Memory Block

In [13]:
class MemoryBlock:
    def __init__(self,
                 state: torch.Tensor | np.ndarray,
                 action: torch.Tensor | np.ndarray,
                 reward: torch.Tensor | np.ndarray,
                 next_state: torch.Tensor | np.ndarray,
                 done: torch.Tensor | np.ndarray,
                 device: Literal["cuda", "cpu"] = None
                 ):
        self.device = device if device else "cuda" if torch.cuda.is_available() else "cpu"
        self.state = self._cast(state)
        self.action = self._cast(action)
        self.reward = self._cast(reward)
        self.next_state = self._cast(next_state)
        self.done = self._cast(done)
        self.to(device)
        self.__check_device()
        
    @staticmethod
    def _cast(x: list | torch.Tensor | np.ndarray) -> torch.Tensor:
        if isinstance(x, (list, float, int, np.number)):
            return torch.tensor(x)
        elif isinstance(x, np.ndarray):
            return torch.from_numpy(x)
        return x

    def to(self, device) -> None:
        self.state = self.state.to(device)
        self.action = self.action.to(device)
        self.reward = self.reward.to(device)
        self.next_state = self.next_state.to(device)
        self.done = self.done.to(device)

    def __check_device(self) -> None:
        shared_device = {
            self.state.device, self.action.device, self.reward.device,
            self.next_state.device, self.done.device
        }
        if len(shared_device) != 1:
            raise AssertionError(
                f"All element in memory block must be in the same device, but got: "
                f"{self.state.device=}, {self.action.device=}, {self.reward.device=}, "
                f"{self.next_state.device=}, {self.done.device}"
            )
        
    def __str__(self) -> str:
        return (
            f"MemoryBlock(\n"
            f"state={self.state!r}\n"
            f"action={self.action!r}\n"
            f"reward={self.reward!r}\n"
            f"next_state={self.next_state!r}\n"
            f"done={self.done!r}\n)"
        )

    def __repr__(self) -> str:
        return (
            f"MemoryBlock("
            f"state={self.state!r}, "
            f"action={self.action!r}, "
            f"reward={self.reward!r}, "
            f"next_state={self.next_state!r}, "
            f"done={self.done!r})"
        )
        

### Memory

In [14]:
class Memory:
    def __init__(self,
                 buffer_size: int, 
                 state_dim: int, 
                 action_dim: int,
                 continuous_state: Optional[bool] = True,
                 continuous_action: Optional[bool] = True,
                 device: Literal["cpu", "cuda"] = None
                 ):
        self.counter = 0
        self.device = device
        self.buffer_size = buffer_size
        self.state_dtype = torch.float32 if continuous_state else torch.int64
        self.action_dtype = torch.float32 if continuous_action else torch.int64
        
        self._states = torch.empty(self.buffer_size, state_dim, dtype=self.state_dtype, device=self.device)
        self._actions = torch.empty(self.buffer_size, action_dim, dtype=self.action_dtype, device=self.device)
        self._rewards = torch.empty(self.buffer_size, 1, dtype=self.state_dtype, device=self.device)
        self._next_states = torch.empty(self.buffer_size, state_dim, dtype=torch.float32, device=self.device)
        self._dones = torch.empty(self.buffer_size, 1, dtype=torch.int64, device=self.device)
        
    def __len__(self) -> int:
        return self.counter
    
    def __getitem__(self, index: int) -> MemoryBlock:
        return MemoryBlock(
            self._states[index], 
            self._actions[index], 
            self._rewards[index], 
            self._next_states[index], 
            self._dones[index]
        )
    
    def __setitem__(self, index: int, block: MemoryBlock) -> None:
        block.to(self.device)
        self._states[index] = block.state
        self._actions[index] = block.action
        self._rewards[index] = block.reward
        self._next_states[index] = block.next_state
        self._dones[index] = block.done
        self.counter += 1

    def clear(self) -> None:
        self._states = torch.empty_like(self._states)
        self._actions = torch.empty_like(self._actions)
        self._rewards = torch.empty_like(self._rewards)
        self._next_states = torch.empty_like(self._next_states)    
        self._dones = torch.empty_like(self._dones)
        self.counter = 0
        
    def write(self, block: MemoryBlock) -> None:
        index = self.counter % self.buffer_size
        self[index] = block


### Replay Buffer

In [15]:
class ReplayBuffer(Memory):
    def __init__(self, 
                 batch_size: int,
                 buffer_size: int, 
                 state_dim: int, 
                 action_dim: int,
                 continuous_state: Optional[bool] = True,
                 continuous_action: Optional[bool] = True,
                 device: Literal["cpu", "cuda"] = None, 
                 ):
        super().__init__(
            buffer_size, state_dim, 
            action_dim, continuous_state, 
            continuous_action, device
        )
        self.batch_size = batch_size
    
    def is_ready(self) -> bool:
        return len(self) >= self.batch_size
    
    def is_full(self) -> bool:
        return len(self) >= self.buffer_size
    
    def sample(self) -> MemoryBlock:
        if self.is_ready():
            indices = torch.randint(
                0, min(len(self), self.buffer_size), 
                (self.batch_size,),
                dtype=torch.int64
            )
            return self[indices]

    def get_dataloader(self, train_steps: Optional[int] = None) -> Generator:
        train_steps = train_steps if train_steps  else len(self) // self.batch_size
        for _ in range(train_steps):
            yield self.sample()
        

## SAC: Soft Actor Critic

In [16]:
class SoftActorCritic(nn.Module):
    def __init__(self,
                 actor: nn.Module,
                 q1_critic: nn.Module,
                 q2_critic: nn.Module, 
                 state_dim: int, 
                 action_dim: int,
                 epoch_per_call: Optional[int] = 1, 
                 train_steps_per_epoch: Optional[int] = 1,
                 update_policy_each_step: Optional[int] = 1,
                 buffer_size: Optional[int] = 2048,
                 batch_size: Optional[int] = 64,
                 p_lr: Optional[float] = 3e-4,
                 q_lr: Optional[float] = 3e-4,
                 gamma: Optional[float] = 0.9,
                 alpha: Optional[float] = 0.005,
                 tau: Optional[float] = 0.01,
                 continuous_state: Optional[bool] = True,
                 continuous_action: Optional[bool] = True,
                 device: Literal["cpu", "cuda"] = None
                 ):
        # ToDO: 
        # - trainable alpha 
        # - soft actor update
        # - action sampling with tanh
        # - starts on buffer warmup
        
        super().__init__()
        self.action_dim = action_dim
        self.device = device if device else "cuda" if torch.cuda.is_available() else "cpu"
        self.buffer = ReplayBuffer(
            batch_size, buffer_size, 
            state_dim, action_dim,
            continuous_state, continuous_action,
            self.device
        )
        
        # Hyper params
        self.epoch_per_call = epoch_per_call
        self.train_steps_per_epoch = train_steps_per_epoch
        self.update_policy_each_step = update_policy_each_step
        self._num_steps = 0
        self.p_lr = p_lr
        self.q_lr = q_lr
        self.alpha = alpha
        self.tau = tau
        self.gamma = gamma
        
        # Models
        self.actor = actor.to(self.device)
        self.q1_critic = q1_critic.to(self.device)
        self.q2_critic = q2_critic.to(self.device)
        self.q1_target = deepcopy(self.q1_critic)
        self.q2_target = deepcopy(self.q2_critic)
        
        # Optim
        self.actor_optim = Adam(self.actor.parameters(), lr=self.p_lr)
        self.q1_optim = Adam(self.q1_critic.parameters(), lr=self.q_lr)
        self.q2_optim = Adam(self.q2_critic.parameters(), lr=self.q_lr)
    
    @torch.no_grad()
    def get_action(self, state: np.ndarray | torch.Tensor) -> np.ndarray:
        state = torch.from_numpy(state) if isinstance(state, np.ndarray) else state
        state = state.to(self.device).unsqueeze(0)
        action, _ = self.forward(state)
        return action.squeeze().cpu().numpy()
    
    def forward(self, state: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        output = self.actor(state)
        mean, log_std = output[:, :self.action_dim], output[:, self.action_dim:]
        dist = Normal(mean, log_std.exp())
        action = dist.rsample()
        log_prob = dist.log_prob(action).mean(dim=-1, keepdim=True)
        return action, log_prob 
    
    def _model_update_step(self, 
               loss: torch.Tensor, 
               optimizer: Optimizer, 
               model: Optional[nn.Module] = None,
               target_model: Optional[nn.Module] = None
               ) -> None:
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    
        if model and target_model and self.tau > 0.:
            for param, target_param in zip(model.parameters(), target_model.parameters()):
                new_target_param = (1. - self.tau) * target_param + self.tau * param
                target_param.data.copy_(new_target_param)
    
    @staticmethod     
    def _unite_state_action(states: torch.Tensor, actions: torch.Tensor) -> torch.Tensor:
        return torch.cat((states, actions), dim=1)

    @torch.no_grad()
    def _compute_q_targets(self, 
                    next_state: torch.Tensor, 
                    reward: torch.Tensor,
                    done: torch.Tensor
                    ) -> torch.Tensor:
        next_action, next_log_prob = self.forward(next_state)
        next_state_action = self._unite_state_action(next_state, next_action)
        q1_values = self.q1_target(next_state_action)
        q2_values = self.q2_target(next_state_action)
        conservative_q_values = torch.min(q1_values, q2_values)
        return reward + self.gamma * (conservative_q_values - self.alpha * next_log_prob) * (1 - done)
    
    def _critic_train_step(self, 
                    states: torch.Tensor, 
                    actions: torch.Tensor, 
                    rewards: torch.Tensor, 
                    next_states: torch.Tensor,
                    dones: torch.Tensor
                    ) -> None:
        targets = self._compute_q_targets(next_states, rewards, dones)
        states_actions = self._unite_state_action(states, actions)
        q1_values = self.q1_critic(states_actions)
        q2_values = self.q2_critic(states_actions)
        q1_loss = F.mse_loss(q1_values, targets)
        q2_loss = F.mse_loss(q2_values, targets)
        self._model_update_step(q1_loss, self.q1_optim, self.q1_critic, self.q1_target)
        self._model_update_step(q2_loss, self.q2_optim, self.q2_critic, self.q2_target)
        
    def _actor_train_step(self, states: torch.Tensor) -> None:
        actions, log_probs = self.forward(states)
        states_actions = self._unite_state_action(states, actions)
        q1_values = self.q1_critic(states_actions)
        q2_values = self.q2_critic(states_actions)
        conservative_q_values = torch.min(q1_values, q2_values)
        actor_loss = - torch.mean(conservative_q_values - self.alpha * log_probs)
        if self._num_steps % self.update_policy_each_step == 0:
            self._model_update_step(actor_loss, self.actor_optim)
    
    def _train_step(self, batch: MemoryBlock) -> None:
        # Fetch memory batch from buffer
        states = batch.state
        next_states = batch.next_state 
        actions = batch.action
        rewards = batch.reward
        dones = batch.done
        
        # Update Critic (Q1, Q2 and Targets Q1 and Q2 | 4 Networks)
        self._critic_train_step(
            states, actions, rewards, 
            next_states, dones
        )
        
        # Update Actor (Policy | 1 Networks)
        self._actor_train_step(states)
        self._num_steps += 1
            
    def fit(self, data: MemoryBlock) -> None:
        self.buffer.write(data) 
        if self.buffer.is_ready():
            for _ in range(self.epoch_per_call):
                for batch in self.buffer.get_dataloader(self.train_steps_per_epoch):
                    self._train_step(batch)

# Let's go to the env | Train

In [17]:
def _env_inner_loop(
        env: gym.Env, 
        agent: nn.Module,
        num_trajectories: int, 
        max_trajectory_len: int = 5_000,
        goal: int = 200
):
    total_rewards = []
    for i in tqdm(range(num_trajectories), desc="Initialize SAC training 🚀"):
        total_reward = 0
    
        state, info = env.reset()
        for _ in range(max_trajectory_len):
            action = agent.get_action(state)
            
            next_state, reward, done, truncated, info = env.step(action)
            total_reward += reward
            
            data = MemoryBlock(
                state, action, reward,
                next_state, done
            )
            agent.fit(data)
            state = next_state
    
            if done:
                break
    
        total_rewards.append(total_reward)
        if len(total_rewards) >= 100 and np.mean(total_rewards[-100:]) >= goal:
            print("Converged 🟩")
            break 
            
        print(f"Total Reward: {total_reward} At {i} trajectory")
    return total_rewards + [total_rewards[-1]] * (num_trajectories - len(total_rewards))


# Достоверность и репрезентативность информации
- Важнейшим требованием к информ при использовании данных является качественн и колличеств однородность
## Посещаемость студентов

На 2 этапе мы с вами расчитываем ассиметрию, обзнач А от х и расчтитывается 

Cредне кважратическое отклонение ассиметрии 

##  5 Среднеквадратическое отклонение эксцесса


Сред. знач 2.56

Сре

Число наблюжений или вероятность наступления событий уменьшается 


# Setup env

In [18]:
env = gym.make(
    "LunarLander-v2", 
    continuous=True, 
    #gravity=-10,
    #enable_wind=False,
    #wind_power=15.0,
    #turbulence_power=1.5,
    render_mode="human"
)

print(env.observation_space.shape, "\n", env.action_space)

(8,) 
 Box(-1.0, 1.0, (2,), float32)


In [19]:
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]
state_dim, action_dim

(8, 2)

In [29]:
policy = nn.Sequential(
    nn.Linear(state_dim, 128),
    nn.ReLU(),
    nn.Linear(128, 256),
    nn.ReLU(),
    nn.Linear(256, action_dim * 2),
    nn.Tanh()
)

q1 = nn.Sequential(
    nn.Linear(state_dim + action_dim, 128),
    nn.ReLU(),
    nn.Linear(128, 1)
)

q2 = nn.Sequential(
    nn.Linear(state_dim + action_dim, 128),
    nn.ReLU(),
    nn.Linear(128, 1)
)


agent = SoftActorCritic(
    actor=policy,
    q1_critic=q1,
    q2_critic=q2,
    state_dim=state_dim,
    action_dim=action_dim,
    buffer_size=100_000,
    gamma=1.,
    tau=0.1,
    alpha=0.01,
    q_lr=1e-3,
    p_lr=1e-3,
    batch_size=1024,
    epoch_per_fit=1,
    train_steps_per_epoch=1,
    update_policy_each_step=1
)

In [ ]:
total_rewards = _env_inner_loop(env, agent, num_trajectories=250, max_trajectory_len=1024)

plt.plot(total_rewards)
plt.show()

Initialize SAC training 🚀:   0%|          | 0/250 [00:00<?, ?it/s]

Total Reward: -203.86112454895346 At 0 trajectory
